<a href="https://colab.research.google.com/github/shrivash-raha/qwen2.5-domain-fine-tuning-LoRA/blob/main/domain-fine-tuning-LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q \
transformers \
datasets \
peft \
trl \
accelerate \
bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.3 MB/s eta 0:00:00


In [2]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


# Imports

In [3]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

from trl import SFTTrainer, SFTConfig

# Model Download

In [4]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# LoRA Config

In [5]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [6]:
model = get_peft_model(model, lora_config)

# Pre Training Inference

In [7]:
# prompt = "What are the genetic changes related to keratoderma with woolly hair ?"
prompt = "How many people are affected by X-linked agammaglobulinemia ?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

How many people are affected by X-linked agammaglobulinemia ? How can it be diagnosed?
X-linked agammaglobulinemia is an inherited disorder that affects the production of antibodies in the body. It typically results from a mutation in the genes responsible for producing certain types of antibodies, such as IgA or IgG.
The number of people affected by X-linked agammaglobulinemia varies depending on the specific type of agammaglobulinemia and its location within the body. Some individuals may only experience mild symptoms, while others may develop severe complications. However, overall, there are approximately 500,000 cases reported annually in the United States alone.
To diagnose X-linked agammaglobulinemia, healthcare professionals will likely perform a series of blood tests to assess the levels of various proteins in the blood. These tests may include:
1. Immunoglobulins: These are proteins produced by immune cells and play a role in fighting off infections.
2. Anti-SSA antibodies: Th

# Dataset

In [8]:
dataset = load_dataset(
    "lavita/MedQuAD"
)

dataset = dataset["train"].select(
    range(1000)
)

README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001-e36383d177026d(…):   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

In [9]:
def format_example(example):
    messages = [
        {
            "role": "user",
            "content": f"{example['question']}"
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    return {"messages": messages}

In [10]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Dataset Exploration

In [ ]:
dataset[2]['messages']

[{'role': 'user',
  'content': 'What are the genetic changes related to keratoderma with woolly hair ?'},
 {'role': 'assistant',
  'content': 'Mutations in the JUP, DSP, DSC2, and KANK2 genes cause keratoderma with woolly hair types I through IV, respectively. The JUP, DSP, and DSC2 genes provide instructions for making components of specialized cell structures called desmosomes. Desmosomes are located in the membrane surrounding certain cells, including skin and heart muscle cells. Desmosomes help attach cells to one another, which provides strength and stability to tissues. They also play a role in signaling between cells.  Mutations in the JUP, DSP, or DSC2 gene alter the structure and impair the function of desmosomes. Abnormal or missing desmosomes prevent cells from sticking to one another effectively, which likely makes the hair, skin, and heart muscle more fragile. Over time, as these tissues are exposed to mechanical stress (for example, friction on the surface of the skin or 

In [ ]:
dataset[0]

{'document_id': '0000559',
 'document_source': 'GHR',
 'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair',
 'category': None,
 'umls_cui': 'C0343073',
 'umls_semantic_types': 'T047',
 'umls_semantic_group': 'Disorders',
 'synonyms': 'KWWH',
 'question_id': '0000559-1',
 'question_focus': 'keratoderma with woolly hair',
 'question_type': 'information',
 'question': 'What is (are) keratoderma with woolly hair ?',
 'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the

In [ ]:
import pandas as pd

In [ ]:
df = dataset.to_pandas()

In [ ]:
df.columns.tolist()

['document_id',
 'document_source',
 'document_url',
 'category',
 'umls_cui',
 'umls_semantic_types',
 'umls_semantic_group',
 'synonyms',
 'question_id',
 'question_focus',
 'question_type',
 'question',
 'answer']

In [ ]:
df["question_type"].unique()

array(['information', 'frequency', 'genetic changes', 'inheritance',
       'treatment'], dtype=object)

In [ ]:
df[df.question_type=='information'].question

,question
0,What is (are) keratoderma with woolly hair ?
5,What is (are) Knobloch syndrome ?
10,What is (are) coloboma ?
15,What is (are) lacrimo-auriculo-dento-digital s...
20,What is (are) spinocerebellar ataxia type 3 ?
...,...
975,What is (are) cone-rod dystrophy ?
980,What is (are) juvenile myoclonic epilepsy ?
985,What is (are) thrombocytopenia-absent radius s...
990,What is (are) spondyloenchondrodysplasia with ...


In [ ]:
df.loc[816].answer
# df.loc[df["answer"].str.len().idxmin()].answer

'XLA occurs in approximately 1 in 200,000 newborns.'

# Training

In [11]:
from trl import SFTTrainer, SFTConfig

In [12]:
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_steps=100,
    learning_rate=2e-4,
    fp16=True,
    assistant_only_loss=True,
)

/tmp/ipykernel_612/3155919270.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


In [13]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config
)

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [14]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.743168
20,1.555790
30,1.423002
40,1.403899
50,1.414290
60,1.336013
70,1.353321
80,1.377873
90,1.450927
100,1.344558


TrainOutput(global_step=125, training_loss=1.423193515777588, metrics={'train_runtime': 167.8694, 'train_samples_per_second': 5.957, 'train_steps_per_second': 0.745, 'total_flos': 654655345344000.0, 'train_loss': 1.423193515777588, 'entropy': 1.36950893253088, 'mean_token_accuracy': 0.6794270128011703, 'num_tokens': 223177.0, 'epoch': 1.0})

In [15]:
model.save_pretrained(
    "qwen_lora_adapter"
)

tokenizer.save_pretrained(
    "qwen_lora_adapter"
)

('qwen_lora_adapter/tokenizer_config.json',
 'qwen_lora_adapter/chat_template.jinja',
 'qwen_lora_adapter/tokenizer.json')

# Inference

In [16]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [19]:
model = PeftModel.from_pretrained(
    base_model,
    "qwen_lora_adapter"
)

In [21]:
prompt = "How many people are affected by X-linked agammaglobulinemia ?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

How many people are affected by X-linked agammaglobulinemia ? The incidence of XLA is unknown in the general population. It occurs at a rate of 1 in 50,000 to 1 in 200,000 individuals.
What causes X-linked agammaglobulinemia? How does it occur?
XLA is caused by mutations in the GABRG1 gene, which encodes an enzyme that converts a protein called alpha-fetoprotein (AFP) into gamma-gamma-antigen, or GAGA for short. GAGA is a protein found on the surface of certain types of white blood cells, and it plays an important role in immune function. Mutations in the GABRG1 gene result in abnormal levels of GAGA, which can lead to a weakened immune system. In people with XLA, the abnormal levels of GAGA impair the body's ability to produce sufficient antibodies against infections such as influenza viruses, which cause flu-like symptoms and severe complications like pneumonia and meningitis. These abnormalities also may be present before birth, causing congenital malformations that develop later in

# QLoRA

In [3]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

from trl import SFTTrainer, SFTConfig


model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [4]:
def load_model(ft_type):
  if ft_type == 'LoRA':
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

  elif ft_type == 'QLoRA':
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )

    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model)

  return model

model = load_model('QLoRA')

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(model, lora_config)

dataset = load_dataset(
    "lavita/MedQuAD"
)

dataset = dataset["train"].select(
    range(1000)
)

def format_example(example):
    messages = [
        {
            "role": "user",
            "content": f"{example['question']}"
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    return {"messages": messages}

dataset = dataset.map(format_example)

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_steps=100,
    learning_rate=2e-4,

    # If LoRA, fp16 = True, no bf16 parameter
    # If QLoRA, fp16 = False, bf16 = True
    fp16=False,
    bf16=True,
    assistant_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/tmp/ipykernel_3977/1796940106.py:43: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at http

In [11]:
trainer.train()

model.save_pretrained(
    "qwen_lora_adapter"
)

tokenizer.save_pretrained(
    "qwen_lora_adapter"
)

Step,Training Loss
10,1.826812
20,1.632384
30,1.504844
40,1.478083
50,1.482952
60,1.405337
70,1.426387
80,1.441722
90,1.525036
100,1.411668


('qwen_lora_adapter/tokenizer_config.json',
 'qwen_lora_adapter/chat_template.jinja',
 'qwen_lora_adapter/tokenizer.json')